# TCC - Análise Preditiva de Risco Acadêmico Escolar
## CRISP-DM: 3. Data Preparation


## O que acontece nesta fase

A preparação transforma arquivos separados em uma base única de aprendizado.

Neste notebook, o foco está nas quatro bases que entram diretamente na modelagem atual: `notas`, `faltas`, `pagamentos` e `professores`. As bases `aluno` e `responsaveis` continuam importantes para contexto e relatórios, mas não fazem parte do núcleo desta etapa de preparação supervisionada.

Fluxo principal:

1. Carrega notas, faltas, pagamentos e professores.
2. Normaliza tipos, datas, notas e ordem das etapas.
3. Agrega faltas por aluno, ano, disciplina e etapa.
4. Agrega pagamentos por aluno e ano letivo.
5. Consolida professores por turma e disciplina.
6. Une tudo na base de notas.
7. Cria atributos históricos, como últimas notas, médias e tendência.
8. Cria os alvos `target_nota_proxima` e `target_risco_proxima`.
9. Aplica o mínimo de histórico por disciplina.
10. Define as colunas que entram nos modelos.





## Carregar bibliotecas e localizar o projeto

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

In [ ]:
def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "school_predictor").exists() and (candidate / "artifacts").exists():
            return candidate.resolve()
    raise RuntimeError("Nao foi possivel localizar a raiz do projeto.")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

## Importar as mesmas funções da pipeline real

Para que a demonstração fique fiel ao sistema, este notebook reutiliza as funções do pacote `school_predictor`. Assim, se a regra de preparação mudar no código, o notebook também deve ser revisado.




In [ ]:
from school_predictor.pipeline.config import PipelinePaths
from school_predictor.pipeline.data import (
    load_source_tables,
    prepare_absences,
    prepare_grades,
    prepare_payments,
    prepare_teachers,
)
from school_predictor.pipeline.dataset import build_prediction_dataset, select_model_columns
from school_predictor.pipeline import dataset as dataset_steps

## Carregar os CSVs de origem

Nesta etapa, os arquivos já devem existir em `artifacts/database/csv/`. Eles são a fronteira pública do projeto; as consultas SQL reais ficam fora do repositório. A função `load_source_tables` carrega aqui apenas as bases usadas diretamente pela modelagem.


### Exemplo fixo do que entrou na preparação

A fase de preparação usa diretamente estas bases centrais da pipeline:

<div style="font-size: 1em; overflow-x: auto;">

| base_usada_na_preparacao | arquivo | linhas | colunas |
| --- | --- | --- | --- |
| notas | media_nota_aluno.csv | 239678 | 20 |
| faltas | faltas_aluno.csv | 168051 | 22 |
| pagamentos | pagamento_aluno.csv | 220090 | 26 |
| professores | professor_disciplina.csv | 1624 | 15 |

</div>


In [ ]:
paths = PipelinePaths.from_project_root(PROJECT_ROOT, min_history=2, mode="demonstracao_data_preparation")
source_tables = load_source_tables(paths)

raw_summary = pd.DataFrame([
    {"base": name, "linhas_originais": len(frame), "colunas_originais": len(frame.columns)}
    for name, frame in source_tables.items()
])
raw_summary

### O que entrou nesta fase

A tabela acima mostra o ponto de partida real da preparação. Neste momento, os dados ainda estão separados por origem e cada base conserva sua própria granularidade.

O que vai mudar ao longo deste notebook é justamente isso: no fim da fase, o projeto deixará de trabalhar com tabelas isoladas e passará a trabalhar com uma observação temporal única por aluno, disciplina e etapa.




## Funções de visualização segura

Amostras exibidas neste notebook mascaram identificadores, nomes, datas pessoais e campos cadastrais. O objetivo é mostrar a forma dos dados, não revelar pessoas.



In [ ]:
def public_preview(frame: pd.DataFrame, rows: int = 5, columns: list[str] | None = None) -> pd.DataFrame:
    preview = frame.head(rows).copy() if columns is None else frame[columns].head(rows).copy()
    return preview.fillna("<ausente>")

## Preparar notas

A base de notas é normalizada para garantir que `ValorMedia`, `NomePeriodo` e a ordem da etapa estejam em formato adequado. A ordem da etapa é extraída de textos como `1 BIMESTRE`, `2 ETAPA` ou equivalentes.


### Exemplo fixo de como as notas ficam após a preparação

Abaixo está uma pequena amostra estática da base de notas já ordenada e com `stage_order` calculado, usando os nomes pseudonimizados da base pública:

<div style="font-size: 1em; overflow-x: auto;">

| NomePeriodo | NomeCurso | NomeSerie | ApelidoTurma | NomeAluno | NomeDisciplina | NomeEtapa | ValorMedia | stage_order |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 2025 | Ensino Médio | 1ª Série | 1ª Série B - Matutino | Aluno 01790 | Arte | 1º BIMESTRE | 10.0 | 1 |
| 2025 | Ensino Médio | 1ª Série | 1ª Série B - Matutino | Aluno 01790 | Arte | 2º BIMESTRE | 8.8 | 2 |
| 2025 | Ensino Médio | 1ª Série | 1ª Série B - Matutino | Aluno 01790 | Arte | 3º BIMESTRE | 8.1 | 3 |
| 2025 | Ensino Médio | 1ª Série | 1ª Série B - Matutino | Aluno 01790 | Arte | 4º BIMESTRE | 10.0 | 4 |
| 2025 | Ensino Médio | 1ª Série | 1ª Série B - Matutino | Aluno 01790 | Biologia | 1º BIMESTRE | 6.8 | 1 |

</div>


In [ ]:
grades = prepare_grades(source_tables["grades"])

grade_columns = [
    "NomePeriodo", "NomeSerie", "ApelidoTurma", "IDAluno", "NomeAluno",
    "NomeDisciplina", "NomeEtapa", "stage_order", "ValorMedia",
]
public_preview(grades, columns=[column for column in grade_columns if column in grades.columns])

### O que aconteceu com a base de notas

Depois de executar esta célula, a base de notas deixa de ser apenas texto bruto e passa a ter ordem temporal interpretável. `NomePeriodo` vira referência numérica, `ValorMedia` passa a estar coerente para cálculo e `stage_order` cria a sequência das etapas.

Na prática, esse é o primeiro passo que permite falar em histórico do aluno em vez de linhas soltas de nota.




## Preparar faltas

A base de faltas também recebe `stage_order`. Depois, eventos individuais são agregados para virar atributos de modelagem: faltas da etapa e faltas acumuladas. O agrupamento é feito por aluno, período, disciplina e ordem da etapa.




### Exemplo fixo de como as faltas ficam após a agregação

Aqui a falta deixa de ser evento isolado e passa a virar contagem por etapa e acumulado:

<div style="font-size: 1em; overflow-x: auto;">

| NomeAluno | NomePeriodo | NomeDisciplina | stage_order | faltas_etapa | faltas_acumuladas |
| --- | --- | --- | --- | --- | --- |
| Aluno 01790 | 2025 | Arte | 2 | 1 | 1 |
| Aluno 01790 | 2025 | Arte | 3 | 2 | 3 |
| Aluno 01790 | 2025 | Biologia | 1 | 3 | 3 |
| Aluno 01790 | 2025 | Biologia | 2 | 4 | 7 |
| Aluno 01790 | 2025 | Biologia | 3 | 7 | 14 |

</div>



In [ ]:
absences = prepare_absences(source_tables["absences"])
absence_features = dataset_steps._build_absence_features(absences)

display(pd.DataFrame([
    {"etapa": "faltas_normalizadas", "linhas": len(absences), "colunas": len(absences.columns)},
    {"etapa": "faltas_agregadas", "linhas": len(absence_features), "colunas": len(absence_features.columns)},
]))
public_preview(absence_features, columns=["IDAluno", "NomePeriodo", "NomeDisciplina", "stage_order", "faltas_etapa", "faltas_acumuladas"])

### O que mudou nas faltas após a agregação

Antes, cada linha representava um evento de falta. Depois desta transformação, a base passa a registrar quantidade de faltas por etapa e acumulado por disciplina.

Essa mudança é importante porque o modelo não aprende melhor com milhares de eventos isolados; ele aprende melhor com um sinal consolidado que descreve o comportamento do aluno até aquele momento.




## Preparar pagamentos

Pagamentos são normalizados para tipos numéricos, booleanos e datas. Depois são agregados por aluno e ano letivo, evitando que o modelo trabalhe com cada movimento financeiro isolado. Dessa etapa saem indicadores como quantidade de pagamentos, pendências, proporção de mensalidades e média de dias até pagamento.



### Exemplo fixo do resumo financeiro

Aqui o dado financeiro já não é mais transacional: ele vira um resumo anual por aluno:

<div style="font-size: 1em; overflow-x: auto;">

| NomeAluno | NomePeriodo | pagamentos_registrados_ano | pagamentos_pendentes_ano | proporcao_mensalidades_ano | media_valor_pagamento_ano | media_dias_ate_pagamento_ano |
| --- | --- | --- | --- | --- | --- | --- |
| Aluno 01790 | 2025 | 13 | 0 | 0.8461538461538461 | 1736.1538461538462 | 0.0 |
| Aluno 01790 | 2026 | 13 | 9 | 0.8461538461538461 | 1981.5384615384614 | 0.23076923076923078 |
| Aluno 02897 | 2025 | 14 | 0 | 0.7857142857142857 | 1382.142857142857 | 0.0 |
| Aluno 02897 | 2026 | 16 | 9 | 0.6875 | 1625.9375 | 0.1875 |
| Aluno 05635 | 2025 | 14 | 0 | 0.7857142857142857 | 1382.142857142857 | 0.0 |

</div>


In [ ]:
payments = prepare_payments(source_tables["payments"])
payment_features = dataset_steps._build_payment_features(payments)

display(pd.DataFrame([
    {"etapa": "pagamentos_normalizados", "linhas": len(payments), "colunas": len(payments.columns)},
    {"etapa": "pagamentos_agregados", "linhas": len(payment_features), "colunas": len(payment_features.columns)},
]))
public_preview(payment_features)

### O que mudou nos pagamentos após a preparação

A base financeira deixa de ser uma lista de movimentos individuais e passa a virar contexto anual por aluno. O resultado esperado aqui é um conjunto mais compacto de indicadores: quantidade de registros, pendências, proporção de mensalidades e média de atraso ou antecipação.

Com isso, o dado financeiro passa a ser utilizável pela modelagem sem expor cada pagamento individualmente.




## Preparar professores

Os vínculos de professor são consolidados por unidade, período, curso, série, turma e disciplina. Quando existe mais de um professor, os nomes são agrupados e a quantidade de professores é registrada. Quando não existe professor vinculado, o preenchimento final será feito depois dentro do dataset com um marcador de ausência legítima.



### Exemplo fixo dos vínculos de professor

A preparação consolida o vínculo por turma e disciplina para permitir o merge posterior:

<div style="font-size: 1em; overflow-x: auto;">

| NomePeriodo | NomeSerie | ApelidoTurma | NomeDisciplina | NomeFuncionario | quantidade_professores_disciplina |
| --- | --- | --- | --- | --- | --- |
| 2022 | 7º Ano | 7º Ano B - Matutino | Ciências da Natureza | Professor 00136 | 1 |
| 2022 | 7º Ano | 7º Ano B - Matutino | Educação Física | Professor 00205 | 1 |
| 2022 | 7º Ano | 7º Ano B - Matutino | Matemática | Professor 00022 | 1 |
| 2022 | 7º Ano | 7º Ano B - Matutino | Arte | Professor 00150 | 1 |
| 2022 | 7º Ano | 7º Ano B - Matutino | Noções de Informática | Professor 00072 | 1 |

</div>


In [ ]:
teachers = prepare_teachers(source_tables["teachers"])

display(pd.DataFrame([{"etapa": "professores_consolidados", "linhas": len(teachers), "colunas": len(teachers.columns)}]))
public_preview(teachers)

### O que aconteceu com os vínculos de professor

Depois desta célula, vários registros detalhados podem se tornar um único vínculo consolidado por unidade, período, curso, série, turma e disciplina. Quando há mais de um professor, os nomes são unidos e a quantidade de docentes fica registrada.

Ou seja, o dado do professor deixa de ser apenas cadastro e vira contexto pedagógico pronto para entrar no dataset final.



## Construir o dataset temporal de modelagem

A função `build_prediction_dataset` une tudo e cria as variáveis históricas. O parâmetro `min_history` define quantas observações anteriores o aluno precisa ter naquela disciplina para entrar no dataset final.

Na pipeline operacional, esse valor muda conforme o modo: `previsao_nota` usa 1 por padrão e `alerta_risco` usa 2. Neste notebook, usamos 2 apenas como demonstração mais conservadora, porque ela evidencia melhor o corte de histórico mínimo.





### Exemplo fixo do dataset integrado

Depois dos merges, cada linha já combina desempenho, frequência, financeiro e professor. A versão fixa abaixo usa nomes curtos para caber melhor na página:

<div style="font-size: 1em; overflow-x: auto;">

| Aluno | Ano | Série | Disciplina | Etapa | Nota | Faltas | Pend. | Prof. | Próx. nota | Risco |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Aluno 01790 | 2025 | 1ª Série | Arte | 2º BIM. | 8.8 | 1 | 0 | Prof. 00207 | 8.1 | 0 |
| Aluno 01790 | 2025 | 1ª Série | Arte | 3º BIM. | 8.1 | 3 | 0 | Prof. 00207 | 10.0 | 0 |
| Aluno 01790 | 2025 | 1ª Série | Biologia | 2º BIM. | 7.0 | 7 | 0 | Prof. 00019 | 7.4 | 0 |
| Aluno 01790 | 2025 | 1ª Série | Biologia 1 | 2º BIM. | 7.0 | 7 | 0 | Prof. 00047 | 5.6 | 1 |

</div>

In [ ]:
MODE_DEFAULTS = {
    "previsao_nota": 1,
    "alerta_risco": 2,
}
DEMO_MIN_HISTORY = 2
modeling_dataset = build_prediction_dataset(source_tables, min_history=DEMO_MIN_HISTORY)

pd.DataFrame([{
    "min_history_demonstracao": DEMO_MIN_HISTORY,
    "padrao_previsao_nota": MODE_DEFAULTS["previsao_nota"],
    "padrao_alerta_risco": MODE_DEFAULTS["alerta_risco"],
    "linhas_dataset_modelagem": len(modeling_dataset),
    "colunas_dataset_modelagem": len(modeling_dataset.columns),
    "alunos": modeling_dataset["IDAluno"].nunique() if "IDAluno" in modeling_dataset.columns else None,
    "disciplinas": modeling_dataset["NomeDisciplina"].nunique() if "NomeDisciplina" in modeling_dataset.columns else None,
    "anos_letivos": modeling_dataset["NomePeriodo"].nunique() if "NomePeriodo" in modeling_dataset.columns else None,
}])

### O que nasceu quando o dataset final foi montado

A tabela acima já não representa mais um CSV bruto. Ela representa a base de modelagem propriamente dita. A partir daqui, cada linha tem contexto histórico suficiente para ensinar algo sobre a próxima etapa do aluno.

Isso significa que várias linhas das bases originais ficaram de fora por dois motivos esperados:
- não havia próxima nota conhecida para virar alvo;
- o histórico mínimo da disciplina ainda não era suficiente.




## Atributos históricos criados

Nesta etapa, cada linha passa a carregar não apenas a nota atual, mas também sinais de histórico e contexto. Exemplos: última nota, médias históricas, tendência, faltas acumuladas, comparação com a coorte, contexto financeiro e informações de professor.



In [ ]:
history_columns = [
    "IDAluno", "NomePeriodo", "NomeSerie", "NomeDisciplina", "NomeEtapa",
    "ValorMedia", "nota_anterior_1", "nota_anterior_2", "media_duas_ultimas_notas",
    "media_historica_disciplina", "tendencia_ultima_nota", "faltas_acumuladas",
    "pagamentos_pendentes_ano", "NomeFuncionario", "professor_disponivel",
]

public_preview(modeling_dataset, columns=[column for column in history_columns if column in modeling_dataset.columns])

### O que os atributos históricos passam a mostrar

A saída acima é a virada mais importante desta fase. O registro atual do aluno agora carrega memória: notas anteriores, médias históricas, tendência, faltas acumuladas, contexto financeiro e professor.

Em termos didáticos, o dado deixa de ser uma fotografia isolada e passa a ser uma sequência resumida em atributos tabulares.




## Alvos de previsão

A preparação também cria os alvos usados na modelagem:

- `target_nota_proxima`: a próxima nota real conhecida na sequência aluno x disciplina.
- `target_risco_proxima`: 1 quando a próxima nota fica abaixo de 6.0, caso contrário 0.

Isso permite que o projeto tenha dois tipos de resposta: previsão numérica da nota e alerta pedagógico de risco.




### Exemplo fixo dos alvos criados

Aqui a linha atual passa a carregar o que queremos prever na próxima observação:

<div style="font-size: 1em; overflow-x: auto;">

| NomeAluno | NomeDisciplina | ValorMedia | target_nota_proxima | target_risco_proxima | historico_disciplina_count |
| --- | --- | --- | --- | --- | --- |
| Aluno 01790 | Arte | 8.8 | 8.1 | 0 | 1 |
| Aluno 01790 | Arte | 8.1 | 10.0 | 0 | 2 |
| Aluno 01790 | Biologia | 7.0 | 7.4 | 0 | 1 |
| Aluno 01790 | Biologia | 7.4 | 7.0 | 0 | 2 |
| Aluno 01790 | Biologia 1 | 7.0 | 5.6 | 1 | 1 |

</div>


In [ ]:
target_columns = [
    "IDAluno", "NomePeriodo", "NomeDisciplina", "NomeEtapa", "ValorMedia",
    "target_nota_proxima", "target_risco_proxima",
    "baseline_ultima_nota", "baseline_media_duas_ultimas", "baseline_hibrido",
]

public_preview(modeling_dataset, columns=[column for column in target_columns if column in modeling_dataset.columns])

### O que aconteceu quando os alvos foram criados

Depois desta etapa, o projeto passa a ter duas respostas supervisionadas na mesma linha:
- um valor numérico para a próxima nota;
- um indicador binário dizendo se essa próxima nota caiu abaixo de 6.0.

Esse é o momento em que a base deixa de ser apenas descritiva e passa a ser adequada para regressão e classificação.



## Selecionar colunas para os modelos

Nem toda coluna do dataset entra no treinamento. Identificadores diretos e alvos ficam fora das features. O modelo recebe atributos históricos, contextuais, numéricos e categóricos. Na implementação atual, `NomeSerie`, `NomeDisciplina`, `NomeEtapa` e `NomeFuncionario` entram como categóricas.




### Exemplo fixo das features que seguem para os modelos

Em vez de listar só o começo da lista, a tabela abaixo resume melhor os grupos de atributos usados na modelagem:

<div style="font-size: 1em; overflow-x: auto;">

| grupo | exemplos |
| --- | --- |
| contexto escolar | NomePeriodo, NomeSerie, NomeDisciplina, NomeEtapa |
| desempenho recente | ValorMedia, nota_anterior_1, nota_anterior_2, nota_anterior_3 |
| tendência e histórico | media_historica_disciplina, tendencia_ultima_nota, desvio_historico_disciplina |
| frequência | faltas_etapa, faltas_acumuladas |
| professor | NomeFuncionario, quantidade_professores_disciplina, professor_disponivel |
| financeiro | pagamentos_registrados_ano, pagamentos_pendentes_ano, media_valor_pagamento_ano |

</div>


In [ ]:
feature_columns, categorical_columns = select_model_columns(modeling_dataset)

pd.DataFrame({
    "coluna": feature_columns,
    "tipo_para_modelo": ["categorica" if column in categorical_columns else "numerica_ou_ordinal" for column in feature_columns],
})

### O que sai para os modelos

A tabela acima mostra que nem tudo que existe no dataset vai para o treinamento. Identificadores diretos e colunas-alvo ficam de fora. O que segue adiante são apenas as features com potencial de explicar a próxima nota ou o risco.

Essa separação é importante para evitar vazamento de informacao e para deixar claro o que realmente alimenta a modelagem.




## Tabela final entregue ao Modeling

A tabela final desta fase representa a ponte direta entre a etapa 3 e a etapa 4 do CRISP-DM. Ela usa exatamente as colunas selecionadas por `select_model_columns`, acrescentando os dois alvos supervisionados e os baselines usados na comparação honesta da modelagem.

Por segurança e rastreabilidade, a amostra demonstrativa abaixo segue o mesmo estilo das tabelas fixas deste notebook: mostra a forma dos dados tratados, mas não inclui identificadores diretos como `IDAluno`, `NomeAluno`, `IDMatricula` ou `IDMedia`.




### Exemplo fixo da tabela final que segue para Modeling

Esta é uma pequena amostra da tabela final já tratada. Ela reúne, na mesma linha, o contexto usado como entrada do modelo, os alvos supervisionados e o baseline de comparação.

<div style="font-size: 1.0em; overflow-x: auto;">

| Ano | Série | Disciplina | Etapa | Nota | Nota ant. | Média hist. | Faltas | Pend. | Prof. | Próx. nota | Risco | Baseline |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 2021 | 6º Ano | Arte | 3º BIM. | 7.3 | 9.1 | 7.6 | 0 | 1 | 0 | 8.1 | 0 | 7.5 |
| 2021 | 6º Ano | Ciências da Natureza | 3º BIM. | 9.7 | 8.7 | 8.45 | 0 | 1 | 0 | 8.9 | 0 | 8.69 |
| 2021 | 6º Ano | Educação Física | 3º BIM. | 7.8 | 10.0 | 10.0 | 0 | 1 | 0 | 10.0 | 0 | 8.80 |
| 2021 | 6º Ano | Geografia | 3º BIM. | 9.0 | 7.3 | 6.65 | 0 | 1 | 0 | 9.5 | 0 | 8.10 |

</div>



In [ ]:
modeling_target_columns = ["target_nota_proxima", "target_risco_proxima"]
modeling_baseline_columns = [
    "baseline_ultima_nota",
    "baseline_media_duas_ultimas",
    "baseline_media_tres_ultimas",
    "baseline_hibrido",
]

modeling_ready_columns = [
    column
    for column in [*feature_columns, *modeling_target_columns, *modeling_baseline_columns]
    if column in modeling_dataset.columns
]
modeling_ready_table = modeling_dataset[modeling_ready_columns].copy()

pd.DataFrame([{
    "linhas_tabela_final": len(modeling_ready_table),
    "colunas_tabela_final": len(modeling_ready_table.columns),
    "features_usadas_no_modelo": len(feature_columns),
    "colunas_categoricas": len(categorical_columns),
    "alvos_supervisionados": ", ".join(modeling_target_columns),
    "baselines_para_comparacao": len(modeling_baseline_columns),
}])


### Conferencia dinamica da tabela final

A célula abaixo permite conferir, durante a execução local do notebook, a mesma estrutura da amostra fixa com os dados disponíveis em `artifacts/database/csv/`. Nada e salvo fora do notebook nesta etapa.



In [ ]:
modeling_ready_preview_columns = [
    "NomePeriodo",
    "NomeSerie",
    "NomeDisciplina",
    "NomeEtapa",
    "ValorMedia",
    "nota_anterior_1",
    "media_historica_disciplina",
    "faltas_acumuladas",
    "pagamentos_pendentes_ano",
    "professor_disponivel",
    "target_nota_proxima",
    "target_risco_proxima",
    "baseline_hibrido",
]

public_preview(
    modeling_ready_table,
    rows=5,
    columns=[column for column in modeling_ready_preview_columns if column in modeling_ready_table.columns],
)


## Comparar volume antes e depois da preparação

A redução de linhas e esperada. O dataset final remove linhas sem próxima nota conhecida e linhas que não atingem o mínimo de histórico definido. Em outras palavras, a preparação não tenta prever qualquer linha bruta; ela preserva apenas observações que realmente podem ensinar algo sobre a próxima etapa.




In [ ]:
preparation_volume = pd.DataFrame([
    {"etapa": "notas_originais", "linhas": len(source_tables["grades"])},
    {"etapa": "notas_normalizadas", "linhas": len(grades)},
    {"etapa": "faltas_agregadas", "linhas": len(absence_features)},
    {"etapa": "pagamentos_agregados", "linhas": len(payment_features)},
    {"etapa": "professores_consolidados", "linhas": len(teachers)},
    {"etapa": "dataset_modelagem", "linhas": len(modeling_dataset)},
])

preparation_volume

### Como ler a redução de volume

Se o número de linhas caiu, isso não significa perda indevida. Significa filtragem metodológica. O projeto esta descartando observações que não servem para prever a próxima etapa com critério temporal.

Na leitura didática da banca, este quadro mostra exatamente onde o dado foi ficando mais analítico e menos bruto.



## Resultado da fase

Ao final da preparação, o projeto deixa de ter várias tabelas separadas e passa a ter uma base temporal única, pronta para a fase de Modeling. Essa base contém:

- nota atual e histórico anterior;
- faltas da etapa e acumuladas;
- informações financeiras agregadas;
- professor vinculado ou marcador de ausência de vínculo;
- comparação com média da coorte;
- alvo numérico de próxima nota;
- alvo binário de risco acadêmico;
- baselines simples para comparação honesta com os modelos.

Com isso, a fase seguinte pode comparar algoritmos diferentes sem precisar rediscutir a integração das bases, porque a estrutura temporal de entrada já esta definida.



